# 🏠 House Price Prediction — India
## Exploratory Data Analysis (EDA)

**Objective:** Understand the dataset, uncover patterns, detect outliers, and engineer features to improve model performance.

**Dataset:** House Price India (`House Price India.csv`)  
**Rows:** 14,620 | **Columns:** 23 | **Target:** `Price`

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
sns.set_palette('husl')

print('✅ Libraries loaded successfully')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('../data/House Price India.csv')
print(f'Dataset Shape: {df.shape}')
df.head()

## 3. Basic Information

In [ ]:
print('=== Dataset Info ===')
df.info()

In [ ]:
print('=== Statistical Summary ===')
df.describe().T.style.background_gradient(cmap='Blues')

In [ ]:
print('=== Null Values ===')
null_counts = df.isnull().sum()
print(null_counts)
print(f'\n✅ Total null values: {null_counts.sum()} — Dataset is clean!')

In [ ]:
print('=== Duplicate Rows ===')
print(f'Duplicate rows: {df.duplicated().sum()}')

## 4. Target Variable Analysis — Price

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribution
axes[0].hist(df['Price'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Price Distribution (Original)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Price (INR)')
axes[0].set_ylabel('Frequency')

# Log-transformed distribution
axes[1].hist(np.log1p(df['Price']), bins=50, color='coral', edgecolor='white')
axes[1].set_title('Price Distribution (Log Transformed)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Log(Price)')
axes[1].set_ylabel('Frequency')

# Boxplot
axes[2].boxplot(df['Price'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightgreen', color='green'))
axes[2].set_title('Price Boxplot (Outlier View)', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Price (INR)')

plt.tight_layout()
plt.savefig('../data/price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean Price:   ₹{df["Price"].mean():,.0f}')
print(f'Median Price: ₹{df["Price"].median():,.0f}')
print(f'Std Dev:      ₹{df["Price"].std():,.0f}')
print(f'Min Price:    ₹{df["Price"].min():,.0f}')
print(f'Max Price:    ₹{df["Price"].max():,.0f}')
print(f'\nSkewness: {df["Price"].skew():.4f} → Log transform recommended')

## 5. Feature Distributions

In [ ]:
numeric_cols = ['number of bedrooms', 'number of bathrooms', 'living area',
                'lot area', 'number of floors', 'condition of the house',
                'grade of the house', 'Area of the house(excluding basement)',
                'Area of the basement', 'Number of schools nearby', 'Distance from the airport']

fig, axes = plt.subplots(3, 4, figsize=(20, 14))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col], bins=30, color=sns.color_palette('husl', len(numeric_cols))[i], edgecolor='white')
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')

# Hide unused subplot
for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../data/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Categorical Feature Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# Bedrooms vs Price
bedroom_price = df.groupby('number of bedrooms')['Price'].median().reset_index()
axes[0,0].bar(bedroom_price['number of bedrooms'], bedroom_price['Price']/100000, color='steelblue')
axes[0,0].set_title('Median Price by Bedrooms', fontweight='bold')
axes[0,0].set_xlabel('Number of Bedrooms')
axes[0,0].set_ylabel('Median Price (Lakh ₹)')

# Bathrooms vs Price
bathroom_price = df.groupby(df['number of bathrooms'].round())['Price'].median().reset_index()
axes[0,1].bar(bathroom_price['number of bathrooms'], bathroom_price['Price']/100000, color='coral')
axes[0,1].set_title('Median Price by Bathrooms', fontweight='bold')
axes[0,1].set_xlabel('Number of Bathrooms')
axes[0,1].set_ylabel('Median Price (Lakh ₹)')

# Condition vs Price
condition_price = df.groupby('condition of the house')['Price'].median().reset_index()
axes[0,2].bar(condition_price['condition of the house'], condition_price['Price']/100000, color='mediumseagreen')
axes[0,2].set_title('Median Price by House Condition', fontweight='bold')
axes[0,2].set_xlabel('Condition (1=Poor, 5=Excellent)')
axes[0,2].set_ylabel('Median Price (Lakh ₹)')

# Grade vs Price
grade_price = df.groupby('grade of the house')['Price'].median().reset_index()
axes[1,0].bar(grade_price['grade of the house'], grade_price['Price']/100000, color='mediumpurple')
axes[1,0].set_title('Median Price by Grade', fontweight='bold')
axes[1,0].set_xlabel('Grade of House')
axes[1,0].set_ylabel('Median Price (Lakh ₹)')

# Floors vs Price
floor_price = df.groupby('number of floors')['Price'].median().reset_index()
axes[1,1].bar(floor_price['number of floors'].astype(str), floor_price['Price']/100000, color='darkorange')
axes[1,1].set_title('Median Price by Floors', fontweight='bold')
axes[1,1].set_xlabel('Number of Floors')
axes[1,1].set_ylabel('Median Price (Lakh ₹)')

# Waterfront vs Price
waterfront_price = df.groupby('waterfront present')['Price'].median().reset_index()
axes[1,2].bar(['No Waterfront', 'Waterfront'], waterfront_price['Price']/100000, color=['lightcoral', 'dodgerblue'])
axes[1,2].set_title('Median Price: Waterfront vs Non-Waterfront', fontweight='bold')
axes[1,2].set_ylabel('Median Price (Lakh ₹)')

plt.suptitle('Categorical Feature vs Price Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/categorical_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Correlation Analysis

In [ ]:
# Drop non-useful columns for correlation
corr_cols = df.drop(columns=['id', 'Date']).corr()

plt.figure(figsize=(18, 14))
mask = np.triu(np.ones_like(corr_cols, dtype=bool))
sns.heatmap(corr_cols, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, annot_kws={'size': 8})
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top correlations with Price
price_corr = corr_cols['Price'].drop('Price').sort_values(ascending=False)

plt.figure(figsize=(12, 7))
colors = ['steelblue' if v > 0 else 'coral' for v in price_corr.values]
plt.barh(price_corr.index, price_corr.values, color=colors)
plt.axvline(x=0, color='black', linewidth=0.8)
plt.title('Feature Correlation with Price', fontsize=14, fontweight='bold')
plt.xlabel('Pearson Correlation Coefficient')
plt.tight_layout()
plt.savefig('../data/price_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 Positively Correlated Features:')
print(price_corr.head())
print('\nTop 5 Negatively Correlated Features:')
print(price_corr.tail())

## 8. Scatter Plots — Key Features vs Price

In [ ]:
top_features = ['living area', 'grade of the house', 'Area of the house(excluding basement)',
                'number of bathrooms', 'Distance from the airport', 'Number of schools nearby']

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for i, col in enumerate(top_features):
    axes[i].scatter(df[col], df['Price']/100000, alpha=0.3, s=10,
                    color=sns.color_palette('husl', len(top_features))[i])
    # Trend line
    z = np.polyfit(df[col], df['Price']/100000, 1)
    p = np.poly1d(z)
    x_line = np.linspace(df[col].min(), df[col].max(), 100)
    axes[i].plot(x_line, p(x_line), 'r--', linewidth=2, label='Trend')
    axes[i].set_title(f'{col} vs Price', fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Price (Lakh ₹)')
    axes[i].legend()

plt.suptitle('Key Features vs Price (Scatter Plots)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/scatter_plots.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Outlier Detection

In [ ]:
outlier_cols = ['Price', 'living area', 'lot area', 'number of bedrooms',
                'Area of the basement', 'Distance from the airport']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(outlier_cols):
    axes[i].boxplot(df[col], vert=True, patch_artist=True,
                    boxprops=dict(facecolor='lightblue', color='navy'),
                    medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(f'Outliers: {col}', fontweight='bold')
    axes[i].set_ylabel('Value')

plt.suptitle('Outlier Detection — Boxplots', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/outlier_detection.png', dpi=150, bbox_inches='tight')
plt.show()

# IQR-based outlier count
print('=== Outlier Count (IQR Method) ===')
for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    print(f'{col}: {len(outliers)} outliers ({len(outliers)/len(df)*100:.1f}%)')

## 10. Feature Engineering

In [ ]:
df_eng = df.copy()

# 1. House Age
df_eng['house_age'] = 2024 - df_eng['Built Year']

# 2. Is Renovated
df_eng['is_renovated'] = (df_eng['Renovation Year'] > 0).astype(int)

# 3. Years since renovation (0 if not renovated)
df_eng['years_since_renovation'] = df_eng.apply(
    lambda x: 2024 - x['Renovation Year'] if x['Renovation Year'] > 0 else x['house_age'], axis=1
)

# 4. Price per square foot (for analysis only, not used as input feature)
df_eng['price_per_sqft'] = df_eng['Price'] / df_eng['living area']

# 5. Total area (living + basement)
df_eng['total_area'] = df_eng['living area'] + df_eng['Area of the basement']

# 6. Has basement
df_eng['has_basement'] = (df_eng['Area of the basement'] > 0).astype(int)

# 7. Bedroom-bathroom ratio
df_eng['bed_bath_ratio'] = df_eng['number of bedrooms'] / (df_eng['number of bathrooms'] + 1)

print('✅ New features created:')
new_features = ['house_age', 'is_renovated', 'years_since_renovation',
                'price_per_sqft', 'total_area', 'has_basement', 'bed_bath_ratio']
print(df_eng[new_features].describe().T[['mean', 'min', 'max']])

In [ ]:
# Visualize engineered features vs Price
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

eng_features = ['house_age', 'total_area', 'price_per_sqft',
                'bed_bath_ratio', 'is_renovated', 'has_basement']

for i, col in enumerate(eng_features):
    if df_eng[col].nunique() <= 2:
        group = df_eng.groupby(col)['Price'].median()
        axes[i].bar(group.index.astype(str), group.values/100000,
                    color=['lightcoral', 'steelblue'])
        axes[i].set_title(f'{col} vs Median Price', fontweight='bold')
    else:
        axes[i].scatter(df_eng[col], df_eng['Price']/100000, alpha=0.3, s=8, color='steelblue')
        axes[i].set_title(f'{col} vs Price', fontweight='bold')
    axes[i].set_ylabel('Price (Lakh ₹)')
    axes[i].set_xlabel(col)

plt.suptitle('Engineered Features vs Price', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/engineered_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Geographic Analysis

In [ ]:
plt.figure(figsize=(12, 8))
scatter = plt.scatter(df['Longitude'], df['Lattitude'],
                      c=df['Price']/100000, cmap='YlOrRd',
                      alpha=0.5, s=10)
plt.colorbar(scatter, label='Price (Lakh ₹)')
plt.title('Geographic Price Distribution (Lat/Long)', fontsize=14, fontweight='bold')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.tight_layout()
plt.savefig('../data/geographic_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('🌍 Warmer colors = Higher prices')

## 12. EDA Summary & Key Insights

In [ ]:
print('=' * 60)
print('📊 EDA SUMMARY — KEY INSIGHTS')
print('=' * 60)

print(f"""
Dataset Overview:
  • {df.shape[0]} properties, {df.shape[1]} features, 0 null values
  • Price range: ₹{df['Price'].min():,.0f} to ₹{df['Price'].max():,.0f}
  • Median price: ₹{df['Price'].median():,.0f}

Top Findings:
  1. Price is right-skewed → Log transformation improves model fit
  2. 'grade of the house' has highest positive correlation with Price
  3. 'living area' is the strongest area-based predictor
  4. Waterfront properties command significantly higher prices
  5. Properties with higher school count nearby → higher prices
  6. Distance from airport has slight negative correlation with price

Feature Engineering:
  • Created: house_age, is_renovated, years_since_renovation
  • Created: total_area, has_basement, bed_bath_ratio
  
Outliers:
  • Price and living area have notable outliers (handled via log transform)
  • Bedroom outliers (33 bedrooms?) will be capped during preprocessing

Model Strategy:
  • Features to use: living area, grade, bedrooms, bathrooms,
    condition, floors, location (lat/long/postal), schools,
    airport distance, engineered features
  • Target: Log(Price) for Linear Regression
  • Models: Linear Regression vs Random Forest → compare RMSE & R²
""")

# Save processed dataframe for model training
df_eng.to_csv('../data/processed_data.csv', index=False)
print('✅ Processed dataset saved to ../data/processed_data.csv')
print('✅ EDA Complete! Proceed to model training.')